# 🌌 RealMLP-TD × 8 seeds in parallel (Flax NNX, TPU)

A from-scratch **Flax NNX** port of RealMLP-TD — a strong tabular MLP (tuned defaults:
PBLD periodic numerical embeddings, NTK-parametrized linears, an internal ensemble, and
scheduled lr / dropout / label-smoothing).

Seeds are independent, so this trains **8 seeds at once — one per TPU core** via
`jax.pmap`. On a v5e-8 that means an 8-model ensemble for roughly the wall-clock of a
single model. The seeds are averaged into a lower-variance model.

Outputs: `submission.csv`, plus out-of-fold and test class probabilities
(`oof_realmlp_ens.npy`, `test_realmlp_ens.npy`) — handy as a member for stacking.

**Kaggle setup:** accelerator **TPU VM v5e-8**, Internet **On** (to pip-install
flax/optax), and add the `playground-series-s6e6` competition data.

## 1. Install

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flax", "optax"], check=True)
print("installed flax + optax")

## 2. Imports

In [ ]:
import math
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score

print("jax", jax.__version__, "| devices:", jax.devices())
N_SEEDS = jax.device_count()   # one seed per core
print("training", N_SEEDS, "seeds in parallel")

## 3. RealMLP-TD model

In [ ]:
PI = math.pi

CONFIG = dict(
    n_ens=8, embed_dim=7, onehot_thresh=10, hidden_dims=[512, 512, 512],
    dropout=0.05, p_drop_sched="expm4t", add_front_scale=True,
    pbld_hidden_dim=20, pbld_out_dim=5, pbld_freq_scale=5.0, pbld_lr_factor=0.093,
    lr=0.01, mom=0.9, sq_mom=0.98, lr_sched="flat_cos", flat_ratio=0.3,
    first_layer_lr_factor=1.0, first_layer_wd_factor=0.1, lr_scale_mult=10.0,
    lr_bias_mult=0.1, weight_decay=0.013, wd_scale_mult=0.1, wd_bias_mult=0.5,
    grad_clip=1.0, ls_eps=0.04, ls_eps_sched="cos",
    tfms=["median_center", "robust_scale"],
    epochs=6, train_bs=256, eval_bs=10240, seed=42,
)


# ── Feature engineering (faithful port of the reference cell) ──────────────────
# Color pairs as features. The reference used only u-g, u-r; a single-seed CV screen
# showed adding more colors lifts RealMLP (+~0.0017), while redshift ratios / abs-mag
# proxies hurt (redshift's near-zero tail makes them noisy), so only colors are added.
COLOR_PAIRS = [("u", "g"), ("u", "r"), ("g", "r"), ("r", "i"), ("i", "z"), ("g", "i")]
IMPORTANT_COMBOS = sorted([("alpha_cat_", "delta_cat_"), ("u_cat_", "z_cat_")])


def feature_engineering(df, cat_cols, num_cols, cmap, fit=False):
    df = df.copy()
    df["_g_/_redshift"] = (df["g"] / (df["redshift"] + 1e-6)).astype("float32")
    df["_i_/_redshift"] = (df["i"] / (df["redshift"] + 1e-6)).astype("float32")
    for a, b in COLOR_PAIRS:
        df[f"_{a}-{b}"] = (df[a] - df[b]).astype("float32")

    for col in cat_cols:                                   # string categoricals
        if fit:
            codes, uniq = df[col].factorize(); cmap[col] = uniq
        else:
            code_map = {c: i for i, c in enumerate(cmap[col])}
            codes = df[col].map(code_map).fillna(-1).astype("int32")
        df[col] = codes.astype("int32")

    for col in num_cols:                                   # floor-bucketed numerics
        name = f"{col}_cat_"
        if fit:
            codes, uniq = np.floor(df[col]).factorize(); cmap[col] = uniq
        else:
            code_map = {c: i for i, c in enumerate(cmap[col])}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype("int32")
        df[name] = codes.astype("int32")

    for col, bins_list in {"delta": [100, 500]}.items():   # quantile bins
        for n_bins in bins_list:
            name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode="ordinal",
                                      strategy="quantile", subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                cmap[name] = kb
            else:
                binned = cmap[name].transform(df[[col]]).ravel().astype("int32")
            df[name] = binned

    combo_names = []
    for cols in IMPORTANT_COMBOS:
        name = "_".join(cols) + "_"
        combo_names.append(name)
        s = df[cols[0]].astype(str)
        for c in cols[1:]:
            s = s + "_" + df[c].astype(str)
        if fit:
            codes, uniq = pd.factorize(s, sort=False); cmap[name] = uniq
        else:
            code_map = {c: i for i, c in enumerate(cmap[name])}
            codes = s.map(code_map).fillna(-1).astype("int32")
        df[name] = codes.astype("int32")

    new_cat = [c for c in df.columns if c.endswith("_")]
    new_num = [c for c in df.columns if c.startswith("_")]
    return df, new_cat, new_num, combo_names


class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in
                      ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X, y=None):
        if {"median_center", "robust_scale"} & set(self._tfms):
            self._median = np.median(X, axis=0)
            q = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            z = q == 0.0
            q[z] = 0.5 * (X.max(axis=0)[z] - X.min(axis=0)[z])
            self._iqr = 1.0 / (q + 1e-30)
            self._iqr[q == 0.0] = 0.0
        return self

    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for t in self._tfms:
            if t == "median_center":
                X -= self._median[None, :]
            elif t == "robust_scale":
                X *= self._iqr[None, :]
            elif t == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif t == "l2_normalize":
                n = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(n == 0, 1.0, n)
        return X


def make_target_encoder(seed):
    """Version-robust TargetEncoder: sklearn >=1.9 deprecated shuffle/random_state in
    favour of passing a CV splitter as `cv`; <1.9 only accepts an int."""
    import sklearn
    from sklearn.model_selection import KFold
    skl = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
    if skl >= (1, 9):
        return TargetEncoder(cv=KFold(5, shuffle=True, random_state=seed), smooth="auto")
    return TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=seed)


# ── NNX model components ───────────────────────────────────────────────────────
class PReLU(nnx.Module):
    def __init__(self, init=0.25):
        self.a = nnx.Param(jnp.asarray(init, jnp.float32))

    def __call__(self, x):
        return jnp.where(x >= 0, x, self.a[...] * x)


class NTPLinear(nnx.Module):
    def __init__(self, n_ens, in_f, out_f, *, rngs, bias=True):
        self.in_f = in_f
        self.weight = nnx.Param(jax.random.normal(rngs.params(), (n_ens, in_f, out_f)))
        self.bias = nnx.Param(jax.random.normal(rngs.params(), (n_ens, out_f))) if bias else None

    def __call__(self, x):
        x = jnp.einsum("bki,kio->bko", x, self.weight[...]) / jnp.sqrt(self.in_f)
        if self.bias is not None:
            x = x + self.bias[...][None]
        return x


class ScalingLayer(nnx.Module):
    def __init__(self, n_ens, n_features):
        self.scale = nnx.Param(jnp.ones((n_ens, n_features)))

    def __call__(self, x):
        return x * self.scale[...][None]


class PBLDEmbedding(nnx.Module):
    def __init__(self, n_ens, n_features, hidden, out_dim, freq_scale, *, rngs):
        self.out_dim = out_dim
        self.w1 = nnx.Param(jax.random.normal(rngs.params(), (n_ens, n_features, hidden)) * freq_scale)
        self.b1 = nnx.Param(jax.random.uniform(rngs.params(), (n_ens, n_features, hidden), minval=-PI, maxval=PI))
        self.w2 = nnx.Param(jax.random.normal(rngs.params(), (n_ens, n_features, hidden, out_dim - 1)) / math.sqrt(hidden))
        self.b2 = nnx.Param(jnp.zeros((n_ens, n_features, out_dim - 1)))
        self.act = PReLU()

    def __call__(self, x):  # (batch, n_ens, n_features)
        periodic = jnp.cos(2 * PI * (x[..., None] * self.w1[...][None] + self.b1[...][None]))
        transformed = self.act(jnp.einsum("bkfh,kfhd->bkfd", periodic, self.w2[...]) + self.b2[...][None])
        feat = jnp.concatenate([x[..., None], transformed], axis=-1)
        return feat.reshape(x.shape[0], x.shape[1], -1)


class CategoricalFeatureLayer(nnx.Module):
    """Per-ensemble one-hot (dim<=thresh) or embedding (dim>thresh) for categoricals."""
    def __init__(self, n_ens, cat_dims, embed_dim, onehot_thresh, *, rngs):
        self.n_ens = n_ens
        self.oh_idx = [i for i, d in enumerate(cat_dims) if d <= onehot_thresh]
        self.oh_dims = [cat_dims[i] for i in self.oh_idx]
        self.em_idx = [i for i, d in enumerate(cat_dims) if d > onehot_thresh]
        self.embeds = nnx.List([
            nnx.Param(jax.random.normal(rngs.params(), (n_ens, cat_dims[i], embed_dim)))
            for i in self.em_idx
        ])

    def __call__(self, x):  # x: (batch, n_ens, n_cat) int
        feats = []
        for j, i in enumerate(self.oh_idx):
            feats.append(jax.nn.one_hot(x[:, :, i], self.oh_dims[j], dtype=jnp.float32))
        kk = jnp.arange(self.n_ens)[None, :]               # (1, n_ens)
        for emb, i in zip(self.embeds, self.em_idx):
            feats.append(emb[...][kk, x[:, :, i]])         # (batch, n_ens, embed_dim)
        return jnp.concatenate(feats, axis=2) if feats else jnp.zeros((x.shape[0], self.n_ens, 0))


class RealMLP(nnx.Module):
    def __init__(self, output_dim, cat_dims, n_numerical, cfg, *, rngs):
        n_ens = cfg["n_ens"]
        self.n_ens = n_ens
        self.cate = CategoricalFeatureLayer(n_ens, cat_dims, cfg["embed_dim"], cfg["onehot_thresh"], rngs=rngs)
        self.num_embed = PBLDEmbedding(n_ens, n_numerical, cfg["pbld_hidden_dim"],
                                       cfg["pbld_out_dim"], cfg["pbld_freq_scale"], rngs=rngs)
        num_dim = n_numerical * cfg["pbld_out_dim"]
        cat_dim = sum(d if d <= cfg["onehot_thresh"] else cfg["embed_dim"] for d in cat_dims)
        total = num_dim + cat_dim
        self.scale = ScalingLayer(n_ens, total) if cfg["add_front_scale"] else None
        self.linears = nnx.List([])
        in_dim = total
        for k, h in enumerate(cfg["hidden_dims"]):
            self.linears.append(NTPLinear(n_ens, in_dim, h, rngs=rngs))
            in_dim = h
        self.out = NTPLinear(n_ens, in_dim, output_dim, rngs=rngs)

    def __call__(self, x_num, x_cat, drop_rate, key, train):
        x_num = jnp.broadcast_to(x_num[:, None], (x_num.shape[0], self.n_ens, x_num.shape[1]))
        x_cat = jnp.broadcast_to(x_cat[:, None], (x_cat.shape[0], self.n_ens, x_cat.shape[1]))
        x = jnp.concatenate([self.num_embed(x_num), self.cate(x_cat)], axis=2)
        if self.scale is not None:
            x = self.scale(x)
        for i, lin in enumerate(self.linears):
            x = jax.nn.silu(lin(x))
            if train:
                key, sub = jax.random.split(key)
                mask = (jax.random.uniform(sub, x.shape) >= drop_rate).astype(x.dtype)
                x = x * mask / jnp.maximum(1.0 - drop_rate, 1e-6)
        return jax.nn.softmax(self.out(x), axis=2)          # (batch, n_ens, C)


# ── Schedules / optimizer / loss ──────────────────────────────────────────────
def sched_factor(progress, sched, flat_ratio):
    if sched == "constant":
        return jnp.ones_like(progress)
    if sched == "cos":
        return (jnp.cos(PI * progress) + 1) / 2
    if sched == "flat_cos":
        t = jnp.clip((progress - flat_ratio) / (1 - flat_ratio), 0.0, 1.0)
        return jnp.where(progress < flat_ratio, 1.0, (jnp.cos(PI * t) + 1) / 2)
    if sched == "expm4t":
        return jnp.exp(-4 * progress)
    raise ValueError(sched)


def _key_str(k):
    for attr in ("name", "key", "idx"):
        if hasattr(k, attr):
            return str(getattr(k, attr))
    return str(k)


def label_for(path):
    name = "/".join(_key_str(p) for p in path)
    if "num_embed" in name:
        return "pbld"
    if "scale" in name:
        return "scale"
    if "bias" in name:
        return "bias"
    if "linears/0/weight" in name:
        return "first_w"
    return "other_w"


def build_optimizer(model, cfg, total_steps):
    p = cfg
    def lr(mult):
        return lambda step: p["lr"] * mult * sched_factor(step / total_steps, p["lr_sched"], p["flat_ratio"])
    def aw(mult, wd):
        return optax.adamw(learning_rate=lr(mult), b1=p["mom"], b2=p["sq_mom"], weight_decay=wd)
    transforms = {
        "scale":   aw(p["lr_scale_mult"], p["weight_decay"] * p["wd_scale_mult"]),
        "pbld":    aw(p["pbld_lr_factor"], p["weight_decay"]),
        "first_w": aw(p["first_layer_lr_factor"], p["weight_decay"] * p["first_layer_wd_factor"]),
        "other_w": aw(1.0, p["weight_decay"]),
        "bias":    aw(p["lr_bias_mult"], p["weight_decay"] * p["wd_bias_mult"]),
    }
    def label_fn(params):
        return jax.tree_util.tree_map_with_path(lambda path, _: label_for(path), params)
    tx = optax.chain(optax.clip_by_global_norm(p["grad_clip"]),
                     optax.multi_transform(transforms, label_fn))
    return nnx.Optimizer(model, tx, wrt=nnx.Param)


def smooth_ce(y_true, probs, ls, class_w):  # probs: (N, C)
    C = probs.shape[1]
    y = jnp.full(probs.shape, ls / C).at[jnp.arange(probs.shape[0]), y_true].set(1.0 - ls + ls / C)
    per = -(y * jnp.log(jnp.clip(probs, 1e-15, 1.0))).sum(1)
    w = class_w[y_true]
    return (per * w).sum() / w.sum()

## 4. Parallel (pmap-over-seeds) trainer

The whole training run is a pure function — batches via `lax.scan`, best-epoch selection carried functionally — so `jax.pmap` maps one seed per device. Each seed gets its own RNG (init, dropout, data shuffle). Evaluation is batched to bound memory.

In [ ]:
def make_tx(params, cfg, total_steps):
    p = cfg
    def lr(mult):
        return lambda step: p["lr"] * mult * sched_factor(step / total_steps, p["lr_sched"], p["flat_ratio"])
    def aw(mult, wd):
        return optax.adamw(learning_rate=lr(mult), b1=p["mom"], b2=p["sq_mom"], weight_decay=wd)
    transforms = {
        "scale": aw(p["lr_scale_mult"], p["weight_decay"] * p["wd_scale_mult"]),
        "pbld": aw(p["pbld_lr_factor"], p["weight_decay"]),
        "first_w": aw(p["first_layer_lr_factor"], p["weight_decay"] * p["first_layer_wd_factor"]),
        "other_w": aw(1.0, p["weight_decay"]),
        "bias": aw(p["lr_bias_mult"], p["weight_decay"] * p["wd_bias_mult"]),
    }
    labels = jax.tree_util.tree_map_with_path(lambda path, _: label_for(path), params)
    return optax.chain(optax.clip_by_global_norm(p["grad_clip"]),
                       optax.multi_transform(transforms, labels))


def bal_acc(y, pred, C):
    correct = (pred == y).astype(jnp.float32)
    tp = jnp.zeros(C).at[y].add(correct)
    sup = jnp.zeros(C).at[y].add(1.0)
    return (tp / jnp.maximum(sup, 1.0)).mean()


def build_train_fn(graphdef, tx, cfg, n_train, n_batches, n_classes):
    bs, epochs = cfg["train_bs"], cfg["epochs"]
    total_steps = epochs * n_batches
    n_ens, eval_bs = cfg["n_ens"], cfg["eval_bs"]

    def apply(params, xn, xc, drop, key, train):
        return nnx.merge(graphdef, params)(xn, xc, drop, key, train)

    def loss_fn(params, xn, xc, y, ls, drop, key, cw):
        probs = apply(params, xn, xc, drop, key, True)
        return smooth_ce(jnp.repeat(y, n_ens), probs.reshape(-1, n_classes), ls, cw)

    def predict(params, xn, xc):                       # batched (scan) -> bounds memory
        n = xn.shape[0]; nb = -(-n // eval_bs)
        xn = jnp.pad(xn, ((0, nb * eval_bs - n), (0, 0)))
        xc = jnp.pad(xc, ((0, nb * eval_bs - n), (0, 0)))
        def f(_, b):
            s = b * eval_bs
            cn = jax.lax.dynamic_slice_in_dim(xn, s, eval_bs, 0)
            cc = jax.lax.dynamic_slice_in_dim(xc, s, eval_bs, 0)
            return None, apply(params, cn, cc, 0.0, jax.random.key(0), False).mean(1)
        _, outs = jax.lax.scan(f, None, jnp.arange(nb))
        return outs.reshape(nb * eval_bs, -1)[:n]

    def train_fn(params, key, Xtr_n, Xtr_c, ytr, Xva_n, Xva_c, yva, Xte_n, Xte_c, cw):
        opt_state = tx.init(params)
        # ONE scan over all epochs*batches keeps the compiled graph small (vs unrolling
        # the epoch loop, which makes TPU compilation blow up). Eval + reshuffle happen
        # at epoch boundaries via lax.cond.
        def step(carry, step_i):
            params, opt_state, key, perm, best_score, best_params = carry
            b = step_i % n_batches
            idx = jax.lax.dynamic_slice_in_dim(perm, b * bs, bs, axis=0)
            progress = step_i / total_steps
            ls = cfg["ls_eps"] * sched_factor(progress, cfg["ls_eps_sched"], cfg["flat_ratio"])
            drop = cfg["dropout"] * sched_factor(progress, cfg["p_drop_sched"], cfg["flat_ratio"])
            key, kd, kperm = jax.random.split(key, 3)
            g = jax.grad(loss_fn)(params, Xtr_n[idx], Xtr_c[idx], ytr[idx], ls, drop, kd, cw)
            updates, opt_state = tx.update(g, opt_state, params)
            params = optax.apply_updates(params, updates)
            at_boundary = (b + 1) == n_batches
            def do_eval(_):
                score = bal_acc(yva, predict(params, Xva_n, Xva_c).argmax(1), n_classes)
                imp = score > best_score
                return (jnp.where(imp, score, best_score),
                        jax.tree.map(lambda bp, p: jnp.where(imp, p, bp), best_params, params))
            best_score, best_params = jax.lax.cond(
                at_boundary, do_eval, lambda _: (best_score, best_params), None)
            perm = jax.lax.cond(at_boundary, lambda: jax.random.permutation(kperm, n_train), lambda: perm)
            return (params, opt_state, key, perm, best_score, best_params), None

        key, k0 = jax.random.split(key)
        perm0 = jax.random.permutation(k0, n_train)
        init = (params, opt_state, key, perm0, jnp.asarray(-1.0), params)
        (*_, best_score, best_params), _ = jax.lax.scan(step, init, jnp.arange(total_steps))
        return predict(best_params, Xva_n, Xva_c), predict(best_params, Xte_n, Xte_c), best_score

    return train_fn

## 5. Data, folds, feature engineering

In [ ]:
from pathlib import Path
_k = Path("/kaggle/input")
_hits = sorted(_k.rglob("train.csv")) if _k.exists() else []
DATA = _hits[0].parent if _hits else Path("../data")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print("data dir:", DATA)

train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")
classes = sorted(train_df["class"].unique())
cmap_y = {c: i for i, c in enumerate(classes)}
y = train_df["class"].map(cmap_y).to_numpy()
n_classes = len(classes)

# Fixed stratified 5-fold split (shuffle, seed 42): reproducible OOF, reusable for stacking.
folds = np.zeros(len(y), dtype=int)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y)):
    folds[va] = i

raw_cat = ["spectral_type", "galaxy_population"]
raw_num = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
state = {}
Xtr, new_cat, new_num, combos = feature_engineering(train_df[raw_cat + raw_num], raw_cat[:], raw_num[:], state, fit=True)
Xte, *_ = feature_engineering(test_df[raw_cat + raw_num], raw_cat[:], raw_num[:], state, fit=False)
cat_cols = sorted(raw_cat + new_cat); num_cols = raw_num + new_num
Xtr = Xtr.reindex(sorted(Xtr.columns), axis=1); Xte = Xte.reindex(sorted(Xte.columns), axis=1)
cat_dims = [int(max(Xtr[c].max(), Xte[c].max()) + 1) for c in cat_cols]
print("train", Xtr.shape, "test", Xte.shape)


def tune_class_weights(y_true, proba, n_rounds=3, grid=None):
    if grid is None:
        grid = np.linspace(0.2, 3.0, 29)
    w = np.ones(proba.shape[1]); best = balanced_accuracy_score(y_true, proba.argmax(1))
    for _ in range(n_rounds):
        for k in range(len(w)):
            bw = w[k]
            for g in grid:
                w[k] = g
                s = balanced_accuracy_score(y_true, (proba * w).argmax(1))
                if s > best: best, bw = s, g
            w[k] = bw
    return w / w.mean(), best

## 6. Train (8 seeds in parallel) over the 5 folds

In [ ]:
cfg = dict(CONFIG)
bs = cfg["train_bs"]
oof = np.zeros((len(Xtr), n_classes))      # seed-averaged out-of-fold
test_probs = np.zeros((len(Xte), n_classes))

for fold in range(5):
    tr = np.where(folds != fold)[0]; va = np.where(folds == fold)[0]
    print(f"=== fold {fold} ===", flush=True)
    enc = make_target_encoder(42)
    tr_te = enc.fit_transform(Xtr.iloc[tr][combos], y[tr]).astype(np.float32)
    va_te = enc.transform(Xtr.iloc[va][combos]).astype(np.float32)
    te_te = enc.transform(Xte[combos]).astype(np.float32)
    pre = NumericalPreprocessor(cfg["tfms"]).fit(np.hstack([Xtr.iloc[tr][num_cols].values.astype(np.float32), tr_te]))
    Xtr_n = pre.transform(np.hstack([Xtr.iloc[tr][num_cols].values.astype(np.float32), tr_te]))
    Xva_n = pre.transform(np.hstack([Xtr.iloc[va][num_cols].values.astype(np.float32), va_te]))
    Xte_n = pre.transform(np.hstack([Xte[num_cols].values.astype(np.float32), te_te]))
    clip = np.array(cat_dims) - 1
    Xtr_c = np.clip(Xtr.iloc[tr][cat_cols].values.astype(np.int64), 0, clip)
    Xva_c = np.clip(Xtr.iloc[va][cat_cols].values.astype(np.int64), 0, clip)
    Xte_c = np.clip(Xte[cat_cols].values.astype(np.int64), 0, clip)
    ytr, yva = y[tr], y[va]
    cw = jnp.asarray(compute_class_weight("balanced", classes=np.arange(n_classes), y=ytr), jnp.float32)

    n_train = (len(ytr) // bs) * bs
    n_batches = n_train // bs

    # one differently-seeded initial model per device (leading axis = N_SEEDS)
    @nnx.vmap(in_axes=0)
    def create(seed_key):
        return RealMLP(n_classes, cat_dims, Xtr_n.shape[1], cfg, rngs=nnx.Rngs(params=seed_key))
    graphdef, params8 = nnx.split(create(jax.random.split(jax.random.key(100 + fold), N_SEEDS)))
    params0 = jax.tree.map(lambda x: x[0], params8)
    tx = make_tx(params0, cfg, n_batches * cfg["epochs"])
    train_fn = build_train_fn(graphdef, tx, cfg, n_train, n_batches, n_classes)

    pm = jax.pmap(train_fn, in_axes=(0, 0) + (None,) * 9)
    tkeys = jax.random.split(jax.random.key(200 + fold), N_SEEDS)
    oof8, test8, scores = pm(params8, tkeys,
        jnp.asarray(Xtr_n), jnp.asarray(Xtr_c), jnp.asarray(ytr),
        jnp.asarray(Xva_n), jnp.asarray(Xva_c), jnp.asarray(yva),
        jnp.asarray(Xte_n), jnp.asarray(Xte_c), cw)
    print(f"  per-seed val bal_acc: {np.asarray(scores).round(4)}", flush=True)
    oof[va] = np.asarray(oof8).mean(0)
    test_probs += np.asarray(test8).mean(0) / 5

## 7. Evaluate, save, submit

In [ ]:
raw = balanced_accuracy_score(y, oof.argmax(1))
weights, tuned = tune_class_weights(y, oof)
print(f"{N_SEEDS}-seed ensemble OOF: raw balanced accuracy={raw:.5f}  tuned={tuned:.5f}")
print(f"class multipliers: {dict(zip(classes, weights.round(3)))}")

np.save(OUT / "oof_realmlp_ens.npy", oof)
np.save(OUT / "test_realmlp_ens.npy", test_probs)
preds = np.asarray(classes)[(test_probs * weights).argmax(1)]
pd.DataFrame({"id": test_df["id"], "class": preds}).to_csv(OUT / "submission.csv", index=False)
print("wrote submission.csv, oof_realmlp_ens.npy, test_realmlp_ens.npy")

## Outputs

- **`submission.csv`** — RealMLP 8-seed ensemble predictions.
- **`oof_realmlp_ens.npy`** / **`test_realmlp_ens.npy`** — out-of-fold and test class
  probabilities (columns sorted GALAXY/QSO/STAR, rows in train/test order), a drop-in
  member for a stacking ensemble.

If this notebook helped, an upvote is appreciated 🙂